# Notebook 2: Spectral CNN Training & Evaluation

**Purpose:** Train `CompactSpectralCNN` (via `FaceFFTPipeline`) on paired real/synthetic
video tensors for a configurable number of epochs, then evaluate with F1 score and
confusion matrix.

This notebook demonstrates the full face-fft pipeline:
1. Load paired `.pt` video tensors
2. Visualise frequency-domain signal (real vs. synthetic)
3. Train the lightweight spectral classifier
4. Plot loss curves and evaluate on held-out test data

> **Quick-start fallback:** If you have no data yet, cells up to the DataLoader
> construction will still run using randomly generated tensors so you can inspect
> the spectral visualisation and model architecture without real data.

## 1. Imports & Configuration

Set `REAL_DIR` and `SYNTH_DIR` to directories containing paired `.pt` tensor files
(one file per video, shape `(C, T, H, W)`). Files are matched by filename, so
`real/vid_001.pt` pairs with `synth/vid_001.pt`.

If these directories do not exist, a dummy dataset of random tensors is generated
automatically so the rest of the notebook remains runnable.

| Variable | Default | Description |
|---|---|---|
| `NUM_EPOCHS` | `5` | Training epochs — increase for a real run |
| `BATCH_SIZE` | `4` | Samples per gradient step |
| `LR` | `1e-3` | AdamW learning rate |
| `SAVE_PATH` | `checkpoints/demo_model.pt` | Where to write the best checkpoint |

In [ ]:
import torch
from torch.utils.data import DataLoader
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np

from face_fft.data.dataset import PairedVideoDataset, split_paired_dataset
from face_fft.models.pipeline import FaceFFTPipeline
from face_fft.features.spectral import SpatiotemporalFFT
from face_fft.training.trainer import Trainer
from face_fft.eval.evaluator import Evaluator

# ── Configuration ──────────────────────────────────────────────────────────────
GENERATOR  = "cogvideox"
REAL_DIR   = Path("../src/face_fft/data/real")    # directory of real .pt tensors (C, T, H, W)
SYNTH_DIR  = Path(f"../src/face_fft/data/synth_{GENERATOR}")   # directory of synth .pt tensors (C, T, H, W)

NUM_EPOCHS  = 5
BATCH_SIZE  = 4
LR          = 1e-3
SAVE_PATH   = Path("checkpoints/demo_model.pt")
DEVICE      = "cuda" if torch.cuda.is_available() else "cpu"

# Reproducibility
torch.manual_seed(42)
np.random.seed(42)

print(f"REAL_DIR  : {REAL_DIR.resolve()}")
print(f"SYNTH_DIR : {SYNTH_DIR.resolve()}")
print(f"DEVICE    : {DEVICE}")
print(f"SAVE_PATH : {SAVE_PATH}")

## 2. Dataset Loading & Inspection

`PairedVideoDataset.from_directories` scans both directories and matches files by name.
Each pair yields two samples: label `0` (real) and label `1` (synthetic), so
`len(dataset) == 2 × number_of_pairs`.

The class balance check below confirms the dataset is 50/50 by construction — any
imbalance indicates missing or unmatched files.

In [ ]:
USE_DUMMY = not (REAL_DIR.exists() and SYNTH_DIR.exists())

if USE_DUMMY:
    print("WARNING: REAL_DIR or SYNTH_DIR not found — using randomly generated tensors as fallback.")
    print("Set REAL_DIR and SYNTH_DIR in the config cell and re-run to use real data.\n")

    import tempfile
    _tmpdir = tempfile.mkdtemp()
    _real_dir  = Path(_tmpdir) / "real"
    _synth_dir = Path(_tmpdir) / "synth"
    _real_dir.mkdir(); _synth_dir.mkdir()

    N_DUMMY = 12
    for i in range(N_DUMMY):
        torch.save(torch.rand(3, 16, 256, 256), _real_dir  / f"vid_{i:03d}.pt")
        torch.save(torch.rand(3, 16, 256, 256), _synth_dir / f"vid_{i:03d}.pt")

    full_dataset = PairedVideoDataset.from_directories(_real_dir, _synth_dir)
else:
    full_dataset = PairedVideoDataset.from_directories(REAL_DIR, SYNTH_DIR)

print(f"Total samples (real + synth): {len(full_dataset)}")
print(f"Pairs                       : {len(full_dataset) // 2}")

sample_video, sample_label = full_dataset[0]
print(f"Sample shape : {sample_video.shape}  (C, T, H, W)")
print(f"Sample label : {sample_label}  (0=Real, 1=Synthetic)")

labels = [full_dataset[i][1] for i in range(len(full_dataset))]
print(f"\nClass balance — Real: {labels.count(0)}, Synthetic: {labels.count(1)}")

## 3. Train / Val / Test Split

Splitting is done at the **pair level** (not the sample level) to prevent data leakage:
the real and synthetic versions of the same source video always end up in the same split.

Default ratio: 80% train · 10% val · 10% test.

In [ ]:
all_pairs = full_dataset.data_pairs

train_pairs, val_pairs, test_pairs = split_paired_dataset(
    all_pairs, train_ratio=0.8, val_ratio=0.1, seed=42
)

print(f"Train pairs : {len(train_pairs)}  ({len(train_pairs)*2} samples)")
print(f"Val pairs   : {len(val_pairs)}  ({len(val_pairs)*2} samples)")
print(f"Test pairs  : {len(test_pairs)}  ({len(test_pairs)*2} samples)")

## 4. Spectral Visualisation

Before training, it is worth confirming that a frequency-domain signal actually exists.
This cell applies `SpatiotemporalFFT` to one real and one synthetic video from the
training set and plots the log-magnitude spectrum at the **temporal centre slice**
(i.e., the spatial FFT of frame `T//2`).

**What to look for:**
- Grid-aligned bright spots in the synthetic spectrum — these are harmonic peaks from
  patch tokenisation or latent-space compression grids
- Stronger high-frequency energy in the synthetic video due to temporal downsampling artifacts
- A broadly similar but noisier real spectrum without the periodic structure

If the two spectra look identical, the spectral hypothesis may not hold for this
video type or resolution and further investigation is needed before training.

In [ ]:
fft = SpatiotemporalFFT(log_scale=True)

real_path  = train_pairs[0]["real"]
synth_path = train_pairs[0]["synthetic"]

real_vid  = torch.load(real_path,  map_location="cpu")   # (C, T, H, W)
synth_vid = torch.load(synth_path, map_location="cpu")   # (C, T, H, W)

with torch.no_grad():
    real_spec  = fft(real_vid.unsqueeze(0)).squeeze(0)   # (C, T, H, W)
    synth_spec = fft(synth_vid.unsqueeze(0)).squeeze(0)  # (C, T, H, W)

# Temporal centre slice of channel 0
t_center    = real_spec.shape[1] // 2
real_slice  = real_spec[0, t_center].numpy()
synth_slice = synth_spec[0, t_center].numpy()

vmin = min(real_slice.min(), synth_slice.min())
vmax = max(real_slice.max(), synth_slice.max())

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

im0 = axes[0].imshow(real_slice,  cmap="viridis", vmin=vmin, vmax=vmax)
axes[0].set_title("Real — FFT magnitude (temporal centre slice)", fontsize=10)
axes[0].axis("off")
plt.colorbar(im0, ax=axes[0])

im1 = axes[1].imshow(synth_slice, cmap="viridis", vmin=vmin, vmax=vmax)
axes[1].set_title("Synthetic — FFT magnitude (temporal centre slice)", fontsize=10)
axes[1].axis("off")
plt.colorbar(im1, ax=axes[1])

plt.suptitle(
    "Spatiotemporal FFT — channel 0, log-magnitude, zero-frequency centred",
    fontsize=11,
)
plt.tight_layout()
plt.show()

print(
    "Look for grid-aligned spikes or periodic harmonic patterns in the synthetic"
    " spectrum that are absent or weaker in the real spectrum."
)

## 5. DataLoader Construction

Wrap each split in a `PairedVideoDataset` and create PyTorch `DataLoader`s.
The training loader shuffles samples; val and test loaders do not.

`num_workers=0` is safe for interactive use. On the HPC cluster, increase this
to match available CPU cores for faster data loading during multi-epoch runs.

In [ ]:
train_dataset = PairedVideoDataset(train_pairs)
val_dataset   = PairedVideoDataset(val_pairs)
test_dataset  = PairedVideoDataset(test_pairs)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

print(f"Train batches : {len(train_loader)}")
print(f"Val batches   : {len(val_loader)}")
print(f"Test batches  : {len(test_loader)}")

## 6. Model Initialisation

`FaceFFTPipeline` composes two modules end-to-end:
1. **`SpatiotemporalFFT`** — 3D FFT → log-magnitude spectrum `(C, T, H, W)`
2. **`CompactSpectralCNN`** — lightweight 3D CNN → binary logit

The model outputs a single logit per sample, used with `BCEWithLogitsLoss` during training.
Parameter count should be in the low thousands — consistent with the lightweight design goal.

In [ ]:
model = FaceFFTPipeline(
    log_scale=True,
    in_channels=3,
    base_channels=16,
    num_classes=1,
).to(DEVICE)

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Model      : FaceFFTPipeline")
print(f"Parameters : {n_params:,}")
print(f"Device     : {DEVICE}")
print()
print(model)

## 7. Training

The `Trainer` class runs the train/validate loop using **AdamW** with weight decay.
After each epoch, validation loss is checked and the best-performing checkpoint is
saved to `SAVE_PATH`.

`trainer.train()` returns a `history` dict with per-epoch `train_loss` and `val_loss`
lists, used in the next cell to plot learning curves.

> With `NUM_EPOCHS = 5` and a small dataset this is a quick sanity check.
> For a meaningful run, set `NUM_EPOCHS = 20` or more.

In [ ]:
SAVE_PATH.parent.mkdir(parents=True, exist_ok=True)

trainer = Trainer(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    device=DEVICE,
    lr=LR,
)

history = trainer.train(num_epochs=NUM_EPOCHS, save_path=str(SAVE_PATH))

print("\nTraining complete.")
print(f"Best checkpoint saved to: {SAVE_PATH}")

## 8. Loss Curves

Plot training and validation loss over epochs to diagnose learning behaviour:

- **Both losses decreasing** — model is learning; consider more epochs
- **Train loss decreasing, val loss flat or rising** — overfitting; reduce model capacity or add regularisation
- **Both losses flat from epoch 1** — possible data issue or learning rate too low

In [ ]:
epochs = range(1, NUM_EPOCHS + 1)

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(epochs, history["train_loss"], marker="o", label="Train loss")
ax.plot(epochs, history["val_loss"],   marker="s", label="Val loss")
ax.set_xlabel("Epoch")
ax.set_ylabel("BCEWithLogits Loss")
ax.set_title("Training & Validation Loss")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Final train loss : {history['train_loss'][-1]:.4f}")
print(f"Final val loss   : {history['val_loss'][-1]:.4f}")

## 9. Test Evaluation

Load the **best checkpoint** (lowest validation loss) saved during training and
run inference on the held-out test set.

`Evaluator.evaluate` returns:
- `f1_score` — harmonic mean of precision and recall (sklearn default: binary, positive class = Synthetic)
- `confusion_matrix` — 2×2 matrix `[[TN, FP], [FN, TP]]` where row = true label, column = predicted

In [ ]:
best_state = torch.load(str(SAVE_PATH), map_location=DEVICE)
model.load_state_dict(best_state)
model.eval()

evaluator = Evaluator(model=model, device=DEVICE)
metrics = evaluator.evaluate(test_loader)

print(f"F1 Score         : {metrics['f1_score']:.4f}")
print(f"Confusion Matrix :")
print(np.array(metrics['confusion_matrix']))

## 10. Confusion Matrix

The heatmap below shows how predictions break down across the two classes.
Rows are true labels, columns are predicted labels.

- **Top-left (TN):** Real videos correctly classified as real
- **Bottom-right (TP):** Synthetic videos correctly classified as synthetic
- **Top-right (FP):** Real videos falsely flagged as synthetic (false alarm)
- **Bottom-left (FN):** Synthetic videos missed (detection failure)

In [ ]:
cm = np.array(metrics["confusion_matrix"])
class_names = ["Real", "Synthetic"]

fig, ax = plt.subplots(figsize=(5, 4))
im = ax.imshow(cm, interpolation="nearest", cmap="Blues")
plt.colorbar(im, ax=ax)

tick_marks = np.arange(len(class_names))
ax.set_xticks(tick_marks)
ax.set_xticklabels(class_names, fontsize=11)
ax.set_yticks(tick_marks)
ax.set_yticklabels(class_names, fontsize=11)

ax.set_xlabel("Predicted label", fontsize=12)
ax.set_ylabel("True label", fontsize=12)
ax.set_title(f"Confusion Matrix (F1 = {metrics['f1_score']:.3f})", fontsize=12)

thresh = cm.max() / 2.0
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        ax.text(
            j, i, format(cm[i, j], "d"),
            ha="center", va="center",
            color="white" if cm[i, j] > thresh else "black",
            fontsize=13,
        )

plt.tight_layout()
plt.show()

## Interpretation Guidance

### What a good F1 looks like

| F1 range | Interpretation |
|---|---|
| 0.90 – 1.00 | Strong spectral separation; model reliably detects generator artifacts |
| 0.75 – 0.90 | Reasonable signal; may improve with more training data or tuning |
| 0.60 – 0.75 | Weak but non-trivial; check spectral visualisation for visible structure |
| < 0.60 | Near-random; possible data issue or mismatched preprocessing |

A **5-epoch demo** on a small dataset will rarely reach the top tier — this notebook
is a sanity check, not a production run. Increase `NUM_EPOCHS` and ensure adequate
paired data for meaningful results.

### Cross-generator notes

- A model trained exclusively on **CogVideoX** data should be re-evaluated on data from
  other generators (e.g., Wan) to assess generalisation.
- Strong in-distribution F1 with weak cross-generator F1 suggests the model has memorised
  generator-specific artifacts rather than learning general compression signatures.
- The spectral visualisation in **Cell 4** is the first diagnostic: if no obvious frequency
  differences are visible between real and synthetic, the spectral hypothesis may not hold
  for this video type or resolution setting.

### Next steps
- Increase training data and re-run with `NUM_EPOCHS = 20` or more
- Test on held-out generators to measure cross-model generalisation
- Inspect misclassified examples to identify failure modes